# Week 5: Dynamic Mapping & Time-Machine Simulation

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Call the CWA real-time rainfall API
2. Parse nested JSON into GeoDataFrame
3. Build interactive Folium maps with conditional styling
4. "Replay" Typhoon Fung-wong (2025) as a stress test
5. Overlay dynamic rainfall with shelter risk data

**Packages needed:** `geopandas`, `folium`, `requests`, `python-dotenv`, `branca`

## Cell [1]: Setup & Load Shelter Data

**What to do:**
- Import all required packages
- Load environment variables from .env
- Load shelter data from Week 3-4 (or create synthetic data)
- Print data summary

In [2]:
# Cell [1]: YOUR CODE HERE
# 1. Import packages: geopandas, pandas, numpy, folium, requests, json, os
# 2. Load .env with python-dotenv
# 3. Load Week 3-4 shelter data (use your ARIA v2 output, or create synthetic data)
# 4. Print shelter count, CRS, and columns

# AI Prompt suggestion:
# "Import geopandas, pandas, numpy, folium, requests, json, os, and dotenv.
#  Create 13 synthetic shelter points in Hualien County (EPSG:3826)
#  with columns: shelter_id, name, TOWNNAME, risk_level, terrain_risk,
#  mean_elevation, max_slope, geometry.
#  Use realistic town names like 花蓮市, 吉安鄉, 壽豐鄉, etc."

import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv
from shapely.geometry import Point
import random

# Load environment variables
load_dotenv()

print("✅ Week 5 Environment Setup Complete")
print(f"GeoPandas: {gpd.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Folium: {folium.__version__}")

✅ Week 5 Environment Setup Complete
GeoPandas: 1.1.3
Pandas: 3.0.1
NumPy: 2.4.2
Folium: 0.20.0


In [2]:
TARGET_COUNTY = os.getenv('TARGET_COUNTY', '花蓮縣')
OUTPUT_DIR = os.getenv('OUTPUT_DIR', 'outputs')
MAP_CENTER_LAT = float(os.getenv('MAP_CENTER_LAT', '23.8'))
MAP_CENTER_LON = float(os.getenv('MAP_CENTER_LON', '121.3'))
MAP_ZOOM = int(os.getenv('MAP_ZOOM', '8'))

print(f"🎯 Week 5 Configuration:")
print(f"   Target County: {TARGET_COUNTY}")
print(f"   Output Directory: {OUTPUT_DIR}")
print(f"   Map Center: ({MAP_CENTER_LAT}, {MAP_CENTER_LON})")
print(f"   Map Zoom: {MAP_ZOOM}")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory ready: {OUTPUT_DIR}")

🎯 Week 5 Configuration:
   Target County: 花蓮縣
   Output Directory: outputs
   Map Center: (23.8, 121.3)
   Map Zoom: 8
✅ Output directory ready: outputs


In [3]:
# 3. Load Week 3-4 shelter data (use your ARIA v2 output, or create synthetic data)

# Try to load ARIA v2 output first
aria_v2_path = '../week4_vector_raster/Homework/outputs/ARIA_v2_shelters_enhanced.gpkg'

if os.path.exists(aria_v2_path):
    print(f"📂 Loading ARIA v2 output from: {aria_v2_path}")
    shelters = gpd.read_file(aria_v2_path)
    print(f"✅ Loaded {len(shelters)} shelters from ARIA v2")
else:
    print(f"⚠️ ARIA v2 output not found, creating synthetic data")
    
    # Create 13 synthetic shelter points in Hualien County (EPSG:3826)
    hualien_towns = [
        '花蓮市', '吉安鄉', '壽豐鄉', '鳳林鎮', '新城鄉',
        '秀林鄉', '光復鄉', '豐濱鄉', '瑞穗鄉', '萬榮鄉',
        '玉里鎮', '卓溪鄉', '富里鄉'
    ]
    
    # Hualien County approximate bounds (EPSG:3826)
    min_x, max_x = 300000, 320000  # Approximate TWD97 x range
    min_y, max_y = 2630000, 2650000  # Approximate TWD97 y range
    
    synthetic_shelters = []
    risk_levels = ['Critical', 'High', 'Medium', 'Low']
    terrain_risks = ['High', 'Medium', 'Low']
    
    for i in range(13):
        # Generate random coordinates within Hualien bounds
        x = random.uniform(min_x, max_x)
        y = random.uniform(min_y, max_y)
        
        # Convert to EPSG:4326 for realistic lat/lon
        temp_gdf = gpd.GeoDataFrame(
            {'geometry': [Point(x, y)]}, 
            crs='EPSG:3826'
        )
        temp_gdf = temp_gdf.to_crs('EPSG:4326')
        
        lon, lat = temp_gdf.geometry[0].x, temp_gdf.geometry[0].y
        
        synthetic_shelters.append({
            'shelter_id': i + 1,
            'name': f'{hualien_towns[i]}避難所',
            'TOWNNAME': hualien_towns[i],
            'risk_level': random.choice(risk_levels),
            'terrain_risk': random.choice(terrain_risks),
            'mean_elevation': round(random.uniform(50, 800), 1),
            'max_slope': round(random.uniform(5, 45), 1),
            'geometry': Point(x, y)
        })
    
    # Create GeoDataFrame
    shelters = gpd.GeoDataFrame(synthetic_shelters, crs='EPSG:3826')
    
    print(f"✅ Created {len(shelters)} synthetic shelters for {TARGET_COUNTY}")

# Save synthetic data for future use
if not os.path.exists(aria_v2_path):
    synthetic_output_path = f'{OUTPUT_DIR}/week5_synthetic_shelters.gpkg'
    shelters.to_file(synthetic_output_path, driver='GPKG')
    print(f"💾 Saved synthetic data to: {synthetic_output_path}")

📂 Loading ARIA v2 output from: ../week4_vector_raster/Homework/outputs/ARIA_v2_shelters_enhanced.gpkg
✅ Loaded 198 shelters from ARIA v2


In [4]:
# 4. Print shelter count, CRS, and columns

print(f"📊 Week 5 Shelter Data Summary:")
print(f"   Total shelters: {len(shelters)}")
print(f"   Coordinate system: {shelters.crs}")
print(f"   Available columns: {list(shelters.columns)}")

# Print risk level distribution
if 'risk_level' in shelters.columns:
    risk_dist = shelters['risk_level'].value_counts().sort_index(ascending=False)
    print(f"\n🎯 Risk Level Distribution:")
    for risk, count in risk_dist.items():
        percentage = count / len(shelters) * 100
        print(f"   {risk}: {count} shelters ({percentage:.1f}%)")

# Print terrain risk distribution
if 'terrain_risk' in shelters.columns:
    terrain_dist = shelters['terrain_risk'].value_counts().sort_index(ascending=False)
    print(f"\n⛰️ Terrain Risk Distribution:")
    for risk, count in terrain_dist.items():
        percentage = count / len(shelters) * 100
        print(f"   {risk}: {count} shelters ({percentage:.1f}%)")

# Print elevation and slope statistics
if 'mean_elevation' in shelters.columns:
    print(f"\n📈 Elevation Statistics:")
    print(f"   Range: {shelters['mean_elevation'].min():.1f} – {shelters['mean_elevation'].max():.1f} m")
    print(f"   Average: {shelters['mean_elevation'].mean():.1f} m")

if 'max_slope' in shelters.columns:
    print(f"\n⛰️ Slope Statistics:")
    print(f"   Range: {shelters['max_slope'].min():.1f}° – {shelters['max_slope'].max():.1f}°")
    print(f"   Average: {shelters['max_slope'].mean():.1f}°")

# Print town distribution
if 'TOWNNAME' in shelters.columns:
    town_dist = shelters['TOWNNAME'].value_counts().sort_index()
    print(f"\n🏘️ Shelter Distribution by Town:")
    for town, count in town_dist.items():
        print(f"   {town}: {count} shelters")

print(f"\n✅ Week 5 data loading and validation complete!")
print(f"📍 Ready for advanced spatial analysis and visualization")

📊 Week 5 Shelter Data Summary:
   Total shelters: 198
   Coordinate system: EPSG:3826
   Available columns: ['shelter_id', 'name', 'capacity', 'indoor', 'river_distance_category', 'mean_elevation', 'max_slope', 'mean_slope', 'std_elevation', 'risk_level', 'geometry']

🎯 Risk Level Distribution:
   Medium: 22 shelters (11.1%)
   Low: 52 shelters (26.3%)
   High: 71 shelters (35.9%)
   Critical: 53 shelters (26.8%)

📈 Elevation Statistics:
   Range: -10210.0 – 1150.6 m
   Average: -57.0 m

⛰️ Slope Statistics:
   Range: 3.8° – 90.0°
   Average: 27.1°

✅ Week 5 data loading and validation complete!
📍 Ready for advanced spatial analysis and visualization


## Cell [2]: Fetch CWA Rainfall API

**What to do:**
- Write a function `fetch_cwa_api(api_key)` that calls the CWA rainfall endpoint
- Handle errors gracefully
- Return JSON response

In [4]:
# Cell [2]: YOUR CODE HERE
# Write a function fetch_cwa_api(api_key) that:
# 1. Calls https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
# 2. Returns the JSON response
# 3. Handles errors with try/except

# AI Prompt suggestion:
# "Write a function fetch_cwa_api(api_key) that calls the CWA rainfall API
#  endpoint O-A0002-001 with requests.get(). Include error handling.
#  The API key goes in the Authorization parameter.
#  Return resp.json() on success, None on failure."

import requests
import json
from datetime import datetime

def fetch_cwa_api(api_key):
    """
    Fetch rainfall data from CWA Open Data API
    
    Args:
        api_key (str): CWA authorization key
        
    Returns:
        dict: JSON response on success, None on failure
    """
    # API endpoint for rainfall data
    url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    
    # Parameters for the API request
    params = {
        'Authorization': api_key,
        'format': 'JSON'
    }
    
    try:
        # Make the API request
        print(f"🌧️ Fetching rainfall data from CWA API...")
        print(f"📍 Endpoint: {url}")
        print(f"🔑 API Key: {api_key[:10]}...{api_key[-4:]}")
        
        response = requests.get(url, params=params, timeout=30)
        
        # Check if request was successful
        response.raise_for_status()
        
        # Parse JSON response
        data = response.json()
        
        # Validate response structure
        if 'success' in data and data['success']:
            print(f"✅ API request successful")
            
            # Extract metadata
            if 'result' in data:
                result = data['result']
                print(f"📊 Retrieved {result.get('total', 'unknown')} records")
                print(f"⏰ Data time: {result.get('resource_id', 'unknown')}")
            
            return data
        else:
            print(f"❌ API returned error: {data.get('message', 'Unknown error')}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error occurred: {e}")
        return None
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
        return None
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return None

# Test the function with example (replace with your actual API key)
# Uncomment the following lines to test:
# test_api_key = "CWA-XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"  # Replace with your key
# rainfall_data = fetch_cwa_api(test_api_key)
# if rainfall_data:
#     print(f"\n📈 Data structure: {list(rainfall_data.keys())}")
# else:
#     print("\n⚠️ Failed to fetch rainfall data")

print("✅ fetch_cwa_api() function ready for use")
print(f"🕐 Created at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ fetch_cwa_api() function ready for use
🕐 Created at: 2026-03-24 16:31:36


In [6]:
# Additional helper function to parse rainfall data
def parse_rainfall_data(api_response):
    """
    Parse rainfall data from CWA API response
    
    Args:
        api_response (dict): JSON response from CWA API
        
    Returns:
        list: List of rainfall station records
    """
    if not api_response or 'records' not in api_response:
        print("❌ No rainfall data found in response")
        return []
    
    records = api_response['records']
    rainfall_stations = []
    
    for record in records:
        try:
            # Extract station information
            station_data = {
                'station_id': record.get('StationID', ''),
                'station_name': record.get('StationName', ''),
                'county': record.get('County', ''),
                'township': record.get('Township', ''),
                'location': {
                    'lat': float(record.get('Latitude', 0)),
                    'lon': float(record.get('Longitude', 0))
                },
                'observations': []
            }
            
            # Extract rainfall observations
            if 'WeatherElement' in record:
                weather_elements = record['WeatherElement']
                if isinstance(weather_elements, list):
                    for element in weather_elements:
                        if element.get('ElementName') == 'Precipitation':
                            observation_data = {
                                'element_name': element.get('ElementName', ''),
                                'element_value': element.get('ElementValue', {})
                            }
                            station_data['observations'].append(observation_data)
            
            rainfall_stations.append(station_data)
            
        except (ValueError, KeyError) as e:
            print(f"⚠️ Error parsing station {record.get('StationID', 'unknown')}: {e}")
            continue
    
    print(f"✅ Parsed {len(rainfall_stations)} rainfall stations")
    return rainfall_stations

# Function to get rainfall for specific county
def get_county_rainfall(api_key, county_name):
    """
    Get rainfall data for a specific county
    
    Args:
        api_key (str): CWA authorization key
        county_name (str): County name (e.g., '花蓮縣')
        
    Returns:
        list: List of rainfall stations in the county
    """
    print(f"🌍 Fetching rainfall data for {county_name}")
    
    # Fetch all rainfall data
    api_response = fetch_cwa_api(api_key)
    
    if not api_response:
        return []
    
    # Parse rainfall data
    all_stations = parse_rainfall_data(api_response)
    
    # Filter by county
    county_stations = [
        station for station in all_stations
        if station['county'] == county_name
    ]
    
    print(f"📍 Found {len(county_stations)} stations in {county_name}")
    return county_stations

print("✅ Additional rainfall parsing functions ready")
print("📊 Available functions:")
print("   - fetch_cwa_api(api_key): Fetch raw API data")
print("   - parse_rainfall_data(response): Parse rainfall stations")
print("   - get_county_rainfall(api_key, county): Get county-specific data")

✅ Additional rainfall parsing functions ready
📊 Available functions:
   - fetch_cwa_api(api_key): Fetch raw API data
   - parse_rainfall_data(response): Parse rainfall stations
   - get_county_rainfall(api_key, county): Get county-specific data


## Cell [3]: Parse Rainfall JSON → GeoDataFrame

**What to do:**
- Extract station data from nested JSON structure
- Filter out invalid data (-998 values)
- Create GeoDataFrame with proper CRS

In [ ]:
# Cell [3]: YOUR CODE HERE
# Write a function parse_rainfall_json(data) that:
# 1. Detects JSON format (CWA API vs CoLife-converted JSON vs XML download)
# 2. Extracts station list from the correct root path
# 3. Gets: StationName, StationId, lat, lon, rain_1hr, rain_3hr, rain_24hr
# 4. Filters out -998 values (NoData sentinel)
# 5. Returns a GeoDataFrame with CRS EPSG:4326

# ⚠️ KEY DIFFERENCES between data sources:
#   - CWA API (format=JSON): records.Station[], Coordinates has 2 sets
#     → [0]=TWD67, [1]=WGS84. You MUST pick WGS84 or stations shift ~1km!
#   - CoLife (originally CSV, instructor converted to JSON):
#     records.Station[], Coordinates has 1 set (WGS84 only)
#   - CWA XML download: cwaopendata.dataset.Station[], 2 coordinate sets
#   - Precipitation: may be str ("130.5") or float (130.5) depending on source
#   - CountyName/TownName: inside GeoInfo, NOT at station top level

# AI Prompt suggestion:
# "Write a normalize_cwa_json(raw) function that detects the JSON format:
#  - If raw has 'records'.'Station', return that list.
#  - If raw has 'cwaopendata'.'dataset'.'Station', return that list.
#  Then write parse_rainfall_json(data) that:
#  - Calls normalize_cwa_json to get the station list
#  - For coordinates: if len(Coordinates) >= 2, find the one with
#    CoordinateName == 'WGS84'; if only 1 entry, use it directly.
#  - Convert Precipitation to float (handles both str and number).
#  - Filter out -998 sentinel values (set to 0).
#  - CountyName and TownName are inside GeoInfo, not at station level.
#  - Return GeoDataFrame with CRS EPSG:4326."

# YOUR CODE HERE

In [10]:
# Cell [3]: 寫一個函數 parse_rainfall_json(data) 用於解析不同來源的雨量資料
# 1. 偵測JSON格式 (CWA API vs CoLife轉換JSON vs XML下載)
# 2. 從正確的根路徑提取站點列表
# 3. 取得: StationName, StationId, lat, lon, rain_1hr, rain_3hr, rain_24hr
# 4. 過濾掉-998值 (NoData哨兵值)
# 5. 返回CRS EPSG:4326的GeoDataFrame

# 修正版本: 解析雨量JSON資料，修正語法錯誤

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import json
from datetime import datetime

def normalize_cwa_json(raw_data):
    """
    正規化CWA JSON格式，偵測並返回站點列表
    
    Args:
        raw_data (dict): 原始JSON資料
        
    Returns:
        list: 站點列表，如果格式不支援則返回空列表
    """
    if not raw_data:
        print("錯誤: 輸入資料為空")
        return []
    
    # 格式1: CWA API (format=JSON) - records.Station[]
    if 'records' in raw_data and 'Station' in raw_data['records']:
        print("偵測到CWA API格式 (records.Station[])")
        return raw_data['records']['Station']
    
    # 格式2: CoLife轉換JSON - records.Station[] (單一座標系統)
    elif 'records' in raw_data and isinstance(raw_data['records'], list):
        print("偵測到CoLife轉換格式 (records[])")
        return raw_data['records']
    
    # 格式3: CWA XML下載轉JSON - cwaopendata.dataset.Station[]
    elif 'cwaopendata' in raw_data and 'dataset' in raw_data['cwaopendata']:
        if 'Station' in raw_data['cwaopendata']['dataset']:
            print("偵測到CWA XML下載格式 (cwaopendata.dataset.Station[])")
            return raw_data['cwaopendata']['dataset']['Station']
    
    # 格式4: 直接是站點列表
    elif isinstance(raw_data, list):
        print("偵測到直接站點列表格式")
        return raw_data
    
    print(f"錯誤: 無法識別的JSON格式。可用鍵值: {list(raw_data.keys())}")
    return []

def parse_rainfall_json(data):
    """
    解析雨量JSON資料，處理多種資料來源格式
    
    Args:
        data (dict): 雨量JSON資料
        
    Returns:
        GeoDataFrame: 包含雨量站資訊的GeoDataFrame (CRS: EPSG:4326)
    """
    print(f"開始解析雨量資料...")
    
    # 步驟1: 正規化JSON格式並取得站點列表
    stations = normalize_cwa_json(data)
    
    if not stations:
        print("錯誤: 無法取得站點列表")
        return gpd.GeoDataFrame()
    
    print(f"取得 {len(stations)} 個站點")
    
    parsed_stations = []
    error_count = 0
    
    for i, station in enumerate(stations):
        try:
            # 基本站點資訊
            station_info = {
                'station_id': None,
                'station_name': None,
                'lat': None,
                'lon': None,
                'county_name': None,
                'town_name': None,
                'rain_1hr': 0.0,
                'rain_3hr': 0.0,
                'rain_24hr': 0.0
            }
            
            # 步驟2: 提取基本站點資訊
            station_info['station_id'] = station.get('StationId', station.get('StationID', f'station_{i}'))
            station_info['station_name'] = station.get('StationName', f'站點_{i}')
            
            # 步驟3: 處理座標系統 (關鍵差異處理)
            coordinates = station.get('Coordinates', [])
            if isinstance(coordinates, list) and len(coordinates) >= 2:
                # 多座標系統情況 (CWA API/XML), 選擇WGS84
                wgs84_coord = None
                for coord in coordinates:
                    if isinstance(coord, dict) and coord.get('CoordinateName') == 'WGS84':
                        wgs84_coord = coord
                        break
                
                if wgs84_coord:
                    station_info['lat'] = float(wgs84_coord.get('Latitude', 0))
                    station_info['lon'] = float(wgs84_coord.get('Longitude', 0))
                    print(f"站點 {station_info['station_name']}: 使用WGS84座標")
                else:
                    print(f"警告: 站點 {station_info['station_name']} 找不到WGS84座標")
                    
            elif isinstance(coordinates, list) and len(coordinates) == 1:
                # 單一座標系統情況 (CoLife格式)
                coord = coordinates[0]
                if isinstance(coord, dict):
                    station_info['lat'] = float(coord.get('Latitude', 0))
                    station_info['lon'] = float(coord.get('Longitude', 0))
                    
            elif isinstance(coordinates, dict):
                # 直接座標物件
                station_info['lat'] = float(coordinates.get('Latitude', 0))
                station_info['lon'] = float(coordinates.get('Longitude', 0))
            
            # 步驟4: 提取行政區資訊 (在GeoInfo內)
            geo_info = station.get('GeoInfo', {})
            if isinstance(geo_info, dict):
                station_info['county_name'] = geo_info.get('CountyName', '')
                station_info['town_name'] = geo_info.get('TownName', '')
            else:
                # 備用: 直接從站點層級尋找
                station_info['county_name'] = station.get('CountyName', '')
                station_info['town_name'] = station.get('TownName', '')
            
            # 步驟5: 處理降雨量資料 (處理字串和數字格式)
            weather_elements = station.get('WeatherElement', [])
            if isinstance(weather_elements, dict):
                weather_elements = [weather_elements]
            
            for element in weather_elements:
                if isinstance(element, dict) and element.get('ElementName') == 'Precipitation':
                    precip_data = element.get('ElementValue', {})
                    
                    # 處理不同格式的降雨量資料
                    def safe_float(value):
                        try:
                            if isinstance(value, str):
                                return float(value)
                            elif isinstance(value, (int, float)):
                                return float(value)
                            else:
                                return 0.0
                        except (ValueError, TypeError):
                            return 0.0
                    
                    # 提取不同時間段的降雨量
                    if isinstance(precip_data, dict):
                        station_info['rain_1hr'] = safe_float(precip_data.get('Past1hr', 0))
                        station_info['rain_3hr'] = safe_float(precip_data.get('Past3hr', 0))
                        station_info['rain_24hr'] = safe_float(precip_data.get('Past24hr', 0))
                    else:
                        station_info['rain_1hr'] = safe_float(precip_data)
            
            # 步驟6: 過濾哨兵值 (-998)
            for rain_key in ['rain_1hr', 'rain_3hr', 'rain_24hr']:
                if station_info[rain_key] == -998 or station_info[rain_key] < 0:
                    station_info[rain_key] = 0.0
            
            # 驗證座標有效性
            if station_info['lat'] and station_info['lon']:
                # 台灣座標範圍檢查
                if 20 <= station_info['lat'] <= 26 and 118 <= station_info['lon'] <= 124:
                    parsed_stations.append(station_info)
                else:
                    print(f"警告: 站點 {station_info['station_name']} 座標超出台灣範圍: ({station_info['lat']}, {station_info['lon']})")
            else:
                print(f"警告: 站點 {station_info['station_name']} 缺少有效座標")
                
        except Exception as e:
            error_count += 1
            print(f"解析站點 {i} 時發生錯誤: {e}")
            continue
    
    print(f"成功解析 {len(parsed_stations)} 個站點，錯誤 {error_count} 個")
    
    # 步驟7: 創建GeoDataFrame
    if parsed_stations:
        df = pd.DataFrame(parsed_stations)
        
        # 創建geometry欄位
        geometry = [Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
        
        print(f"成功創建GeoDataFrame，CRS: {gdf.crs}")
        print(f"欄位: {list(gdf.columns)}")
        
        # 顯示統計資訊
        print(f"\n雨量統計:")
        print(f"  1小時降雨量: {gdf['rain_1hr'].min():.1f} - {gdf['rain_1hr'].max():.1f} mm")
        print(f"  3小時降雨量: {gdf['rain_3hr'].min():.1f} - {gdf['rain_3hr'].max():.1f} mm")
        print(f"  24小時降雨量: {gdf['rain_24hr'].min():.1f} - {gdf['rain_24hr'].max():.1f} mm")
        
        # 顯示縣市分布
        if not gdf['county_name'].isna().all():
            county_count = gdf['county_name'].value_counts()
            print(f"\n縣市分布:")
            for county, count in county_count.head(10).items():
                print(f"  {county}: {count} 個站點")
        
        return gdf
    else:
        print("錯誤: 沒有有效的站點資料")
        return gpd.GeoDataFrame()

print("parse_rainfall_json() 函數已準備就緒")

parse_rainfall_json() 函數已準備就緒


In [11]:
# 測試函數用的範例資料結構 (修正語法錯誤)

# 範例1: CWA API格式 (修正: true -> True)
cwa_api_example = {
    "success": True,  # 修正: 使用Python的True
    "records": {
        "Station": [
            {
                "StationId": "C0A940",
                "StationName": "花蓮",
                "Coordinates": [
                    {
                        "CoordinateName": "TWD67",
                        "Latitude": "23.983333",
                        "Longitude": "121.616667"
                    },
                    {
                        "CoordinateName": "WGS84",
                        "Latitude": 23.983333,
                        "Longitude": 121.616667
                    }
                ],
                "GeoInfo": {
                    "CountyName": "花蓮縣",
                    "TownName": "花蓮市"
                },
                "WeatherElement": [
                    {
                        "ElementName": "Precipitation",
                        "ElementValue": {
                            "Past1hr": "15.5",
                            "Past3hr": 45.2,
                            "Past24hr": "120.8"
                        }
                    }
                ]
            }
        ]
    }
}

# 範例2: CoLife轉換格式
colife_example = {
    "records": [
        {
            "StationID": "C0A941",
            "StationName": "吉安",
            "Coordinates": [
                {
                    "Latitude": 23.966667,
                    "Longitude": 121.583333
                }
            ],
            "GeoInfo": {
                "CountyName": "花蓮縣",
                "TownName": "吉安鄉"
            },
            "WeatherElement": {
                "ElementName": "Precipitation",
                "ElementValue": "8.3"
            }
        }
    ]
}

# 測試解析函數
print("測試CWA API格式:")
cwa_result = parse_rainfall_json(cwa_api_example)
print(f"\n測試CoLife格式:")
colife_result = parse_rainfall_json(colife_example)

if not cwa_result.empty:
    print(f"\nCWA解析結果: {len(cwa_result)} 個站點")
    print(cwa_result[['station_name', 'county_name', 'rain_1hr', 'rain_3hr', 'rain_24hr']].head())

if not colife_result.empty:
    print(f"\nCoLife解析結果: {len(colife_result)} 個站點")
    print(colife_result[['station_name', 'county_name', 'rain_1hr']].head())

測試CWA API格式:
開始解析雨量資料...
偵測到CWA API格式 (records.Station[])
取得 1 個站點
站點 花蓮: 使用WGS84座標
成功解析 1 個站點，錯誤 0 個
成功創建GeoDataFrame，CRS: EPSG:4326
欄位: ['station_id', 'station_name', 'lat', 'lon', 'county_name', 'town_name', 'rain_1hr', 'rain_3hr', 'rain_24hr', 'geometry']

雨量統計:
  1小時降雨量: 15.5 - 15.5 mm
  3小時降雨量: 45.2 - 45.2 mm
  24小時降雨量: 120.8 - 120.8 mm

縣市分布:
  花蓮縣: 1 個站點

測試CoLife格式:
開始解析雨量資料...
偵測到CoLife轉換格式 (records[])
取得 1 個站點
成功解析 1 個站點，錯誤 0 個
成功創建GeoDataFrame，CRS: EPSG:4326
欄位: ['station_id', 'station_name', 'lat', 'lon', 'county_name', 'town_name', 'rain_1hr', 'rain_3hr', 'rain_24hr', 'geometry']

雨量統計:
  1小時降雨量: 8.3 - 8.3 mm
  3小時降雨量: 0.0 - 0.0 mm
  24小時降雨量: 0.0 - 0.0 mm

縣市分布:
  花蓮縣: 1 個站點

CWA解析結果: 1 個站點
  station_name county_name  rain_1hr  rain_3hr  rain_24hr
0           花蓮         花蓮縣      15.5      45.2      120.8

CoLife解析結果: 1 個站點
  station_name county_name  rain_1hr
0           吉安         花蓮縣       8.3


## Cell [4]: Mode Switcher (LIVE vs SIMULATION)

**What to do:**
- Read `APP_MODE` from .env
- If LIVE: fetch real rainfall data from API
- If SIMULATION: load fallback JSON file
- Parse using the same function in both cases

In [ ]:
# Cell [4]: YOUR CODE HERE
# Mode Switcher:
# 1. Read APP_MODE from .env (default: 'SIMULATION')
# 2. If LIVE: call fetch_cwa_api() → parse_rainfall_json()
# 3. If SIMULATION: load fungwong_202511.json → parse_rainfall_json()
# 4. KEY INSIGHT: same parse function for both!

# AI Prompt suggestion:
# "Read APP_MODE from os.getenv (default 'SIMULATION').
#  If LIVE, call fetch_cwa_api and parse the result.
#  If SIMULATION, load 'data/scenarios/fungwong_202511.json' and parse it.
#  Both paths use the same parse_rainfall_json function.
#  Print which mode is active and how many stations loaded."

# YOUR CODE HERE

In [20]:
# 模式切換器 - 整合版本

# 載入環境變數
load_dotenv()

# 讀取應用程式模式
APP_MODE = "LIVE"

print(f"🔧 應用程式模式設定")
print(f"   當前模式: {APP_MODE}")
print(f"   可用模式: LIVE (即時API), SIMULATION (模擬資料)")

# 初始化雨量資料變數
rainfall_gdf = gpd.GeoDataFrame()
data_source = ""
load_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

if APP_MODE == 'LIVE':
    print(f"\n🌐 啟動即時模式 - 從CWA API取得資料")
    
    # 檢查API金鑰
    api_key = os.getenv('CWA_API_KEY')
    if not api_key:
        print("❌ 錯誤: LIVE模式需要設定CWA_API_KEY環境變數")
        print("   請在.env檔案中設定: CWA_API_KEY=your-api-key-here")
        APP_MODE = 'SIMULATION_FALLBACK'
    else:
        print(f"   API金鑰: {api_key[:10]}...{api_key[-4:]}")
        print(f"   正在呼叫CWA API...")
        
        # 呼叫API並解析資料
        api_response = fetch_cwa_api(api_key)
        
        if api_response:
            # 解析API回應
            rainfall_gdf = parse_rainfall_json(api_response)
            data_source = "CWA即時API"
            print(f"   ✅ 成功從CWA API載入資料")
        else:
            print(f"   ❌ API呼叫失敗，切換到模擬模式")
            APP_MODE = 'SIMULATION_FALLBACK'

if APP_MODE in ['SIMULATION', 'SIMULATION_FALLBACK']:
    mode_desc = "模擬模式" if APP_MODE == 'SIMULATION' else "模擬模式 (API失敗備用)"
    print(f"\n📊 啟動{mode_desc} - 載入模擬資料")
    
    # 模擬資料檔案路徑
    simulation_file = 'data/scenarios/fungwong_202511.json'
    
    # 嘗試載入模擬資料
    try:
        # 檢查檔案是否存在
        if os.path.exists(simulation_file):
            print(f"   載入檔案: {simulation_file}")
            
            # 載入JSON檔案
            with open(simulation_file, 'r', encoding='utf-8') as f:
                simulation_data = json.load(f)
            
            # 解析模擬資料
            rainfall_gdf = parse_rainfall_json(simulation_data)
            data_source = simulation_file
            print(f"   ✅ 成功載入模擬資料")
            
        else:
            print(f"   ❌ 模擬資料檔案不存在: {simulation_file}")
            print(f"   創建示範模擬資料...")
            
            # 創建示範模擬資料
            demo_data = {
                "records": [
                    {
                        "StationID": "C0A940",
                        "StationName": "花蓮",
                        "Coordinates": [
                            {
                                "Latitude": 23.983333,
                                "Longitude": 121.616667
                            }
                        ],
                        "GeoInfo": {
                            "CountyName": "花蓮縣",
                            "TownName": "花蓮市"
                        },
                        "WeatherElement": {
                            "ElementName": "Precipitation",
                            "ElementValue": {
                                "Past1hr": "25.5",
                                "Past3hr": 68.2,
                                "Past24hr": "145.8"
                            }
                        }
                    },
                    {
                        "StationID": "C0A941",
                        "StationName": "吉安",
                        "Coordinates": [
                            {
                                "Latitude": 23.966667,
                                "Longitude": 121.583333
                            }
                        ],
                        "GeoInfo": {
                            "CountyName": "花蓮縣",
                            "TownName": "吉安鄉"
                        },
                        "WeatherElement": {
                            "ElementName": "Precipitation",
                            "ElementValue": {
                                "Past1hr": "18.3",
                                "Past3hr": 42.7,
                                "Past24hr": "98.5"
                            }
                        }
                    },
                    {
                        "StationID": "C0A942",
                        "StationName": "壽豐",
                        "Coordinates": [
                            {
                                "Latitude": 23.883333,
                                "Longitude": 121.500000
                            }
                        ],
                        "GeoInfo": {
                            "CountyName": "花蓮縣",
                            "TownName": "壽豐鄉"
                        },
                        "WeatherElement": {
                            "ElementName": "Precipitation",
                            "ElementValue": {
                                "Past1hr": "12.1",
                                "Past3hr": 35.4,
                                "Past24hr": "87.2"
                            }
                        }
                    },
                    {
                        "StationID": "C0A943",
                        "StationName": "新城",
                        "Coordinates": [
                            {
                                "Latitude": 24.016667,
                                "Longitude": 121.650000
                            }
                        ],
                        "GeoInfo": {
                            "CountyName": "花蓮縣",
                            "TownName": "新城鄉"
                        },
                        "WeatherElement": {
                            "ElementName": "Precipitation",
                            "ElementValue": {
                                "Past1hr": "22.7",
                                "Past3hr": 58.9,
                                "Past24hr": "132.4"
                            }
                        }
                    }
                ]
            }
            
            # 解析示範資料
            rainfall_gdf = parse_rainfall_json(demo_data)
            data_source = "示範模擬資料"
            print(f"   ✅ 成功創建並載入示範資料")
            
    except Exception as e:
        print(f"   ❌ 載入模擬資料時發生錯誤: {e}")
        rainfall_gdf = gpd.GeoDataFrame()
        data_source = "載入失敗"

# 顯示載入結果摘要
print(f"\n📊 資料載入摘要")
print(f"   活躍模式: {APP_MODE}")
print(f"   資料來源: {data_source}")
print(f"   載入時間: {load_time}")

if not rainfall_gdf.empty:
    print(f"   載入站點數量: {len(rainfall_gdf)}")
    print(f"   座標系統: {rainfall_gdf.crs}")
    print(f"   可用欄位: {list(rainfall_gdf.columns)}")
    
    # 顯示基本統計
    print(f"\n🌧️ 降雨量統計摘要:")
    print(f"   1小時降雨量: {rainfall_gdf['rain_1hr'].min():.1f} - {rainfall_gdf['rain_1hr'].max():.1f} mm")
    print(f"   3小時降雨量: {rainfall_gdf['rain_3hr'].min():.1f} - {rainfall_gdf['rain_3hr'].max():.1f} mm")
    print(f"   24小時降雨量: {rainfall_gdf['rain_24hr'].min():.1f} - {rainfall_gdf['rain_24hr'].max():.1f} mm")
    
    # 顯示地理分布
    if 'county_name' in rainfall_gdf.columns:
        county_count = rainfall_gdf['county_name'].value_counts()
        print(f"\n🏘️ 縣市分布:")
        for county, count in county_count.head(5).items():
            print(f"   {county}: {count} 個站點")
    
    print(f"\n✅ 資料載入完成，準備進行空間分析")
    
else:
    print(f"   ❌ 資料載入失敗，無法進行後續分析")
    print(f"   請檢查模式設定和資料來源")

# 設定全域變數供後續使用
rainfall_data = rainfall_gdf
data_mode = APP_MODE
data_source_info = data_source

print(f"\n🔧 全域變數已設定:")
print(f"   rainfall_data: 雨量站GeoDataFrame")
print(f"   data_mode: 當前應用模式")
print(f"   data_source_info: 資料來源資訊")

🔧 應用程式模式設定
   當前模式: LIVE
   可用模式: LIVE (即時API), SIMULATION (模擬資料)

🌐 啟動即時模式 - 從CWA API取得資料
   API金鑰: CWA-6AA937...C2EF
   正在呼叫CWA API...
🌧️ Fetching rainfall data from CWA API...
📍 Endpoint: https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
🔑 API Key: CWA-6AA937...C2EF
✅ API request successful
📊 Retrieved unknown records
⏰ Data time: O-A0002-001


ValueError: Assigning CRS to a GeoDataFrame without a geometry column is not supported. Supply geometry using the 'geometry=' keyword argument, or by providing a DataFrame with column name 'geometry'

## Cell [5]: Create Base Folium Map

**What to do:**
- Create a Folium map centered on Hualien County
- Use OpenStreetMap or Satellite basemap
- Set initial zoom level

In [ ]:
# Cell [5]: YOUR CODE HERE
# Create a base Folium map:
# 1. Center on Hualien (latitude ~23.98, longitude ~121.55)
# 2. Use tiles='OpenStreetMap' or tiles='Satellite'
# 3. Set zoom_start=10
# 4. Assign to variable `m`

# AI Prompt suggestion:
# "Create a Folium map centered at [23.98, 121.55] (Hualien County)
#  with zoom_start=10 and tiles='OpenStreetMap'.
#  Store it in variable m."

# YOUR CODE HERE

In [ ]:
# Cell [5]: 創建基礎Folium地圖
# 1. 中心點設定在花蓮縣 (緯度 23.98, 經度 121.55)
# 2. 使用 OpenStreetMap 底圖
# 3. 設定初始縮放級別為 10
# 4. 儲存到變數 m

import folium
import pandas as pd
import geopandas as gpd
from datetime import datetime

print("✅ Folium地圖創建環境準備完成")
print(f"Folium版本: {folium.__version__}")
print(f"創建時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 創建基礎地圖
m = folium.Map(
    location=[23.98, 121.55],  # 花蓮縣中心座標
    zoom_start=10,           # 初始縮放級別
    tiles='OpenStreetMap'    # 使用OpenStreetMap底圖
)

print("\n🗺️ 基礎地圖創建完成")
print(f"   中心點: [23.98, 121.55] (花蓮縣)")
print(f"   縮放級別: 10")
print(f"   底圖樣式: OpenStreetMap")
print(f"   地圖變數: m")

# 顯示地圖基本資訊
print(f"\n📊 地圖資訊:")
print(f"   地圖類型: {type(m)}")
print(f"   預設圖磚: {m.options.get('tiles', 'OpenStreetMap')}")
print(f"   中心座標: {m.location}")

# 顯示地圖
print(f"\n🌍 顯示花蓮縣基礎地圖:")
display(m)

m.save("map2.html")

✅ Folium地圖創建環境準備完成
Folium版本: 0.20.0
創建時間: 2026-03-24 16:20:49

🗺️ 基礎地圖創建完成
   中心點: [23.98, 121.55] (花蓮縣)
   縮放級別: 10
   底圖樣式: OpenStreetMap
   地圖變數: m

📊 地圖資訊:
   地圖類型: <class 'folium.folium.Map'>
   預設圖磚: OpenStreetMap
   中心座標: [23.98, 121.55]

🌍 顯示花蓮縣基礎地圖:


## Cell [6]: Add Rainfall CircleMarkers with Conditional Styling

**What to do:**
- Write a function `rain_color(rain_value)` that returns color based on rainfall amount
- Add CircleMarker for each rainfall station
- Size and color represent rainfall intensity

In [ ]:
# Cell [6]: YOUR CODE HERE
# 1. Write function rain_color(rain_mm) that returns:
#    - 'green'  if rain_mm < 10 mm/hr    (safe)
#    - 'gold'   if 10 <= rain_mm < 40    (caution)
#    - 'orange' if 40 <= rain_mm < 80    (warning)
#    - 'red'    if rain_mm >= 80 mm/hr   (danger)
# 2. Loop through gdf_rainfall and add CircleMarker for each station
# 3. Radius proportional to rain_1hr: radius = max(5, rain_mm / 5)

# AI Prompt suggestion:
# "Write rain_color(rain_mm) that returns 'green' (<10), 'gold' (10-40),
#  'orange' (40-80), 'red' (>=80) based on rainfall intensity.
#  Write rain_radius(rain_mm) that returns max(5, rain_mm / 5).
#  Then add CircleMarker for each station in gdf_rainfall to folium map m.
#  Use the rain_color function to set fill color and rain_radius for size.
#  Add tooltip with station name and rainfall amount."

# YOUR CODE HERE

## Cell [7]: Add HeatMap Layer

**What to do:**
- Import HeatMap from folium.plugins
- Create a heat layer showing rainfall intensity
- Add to folium map

In [ ]:
# Cell [7]: YOUR CODE HERE
# 1. Import HeatMap from folium.plugins
# 2. Create list of [lat, lon, rain_1hr] for each station
# 3. Add HeatMap(data, name='Rainfall Heatmap', show=False) to map m

# AI Prompt suggestion:
# "Import HeatMap from folium.plugins.
#  Create a heat_data list of [lat, lon, rain_1hr] from gdf_rainfall.
#  Add HeatMap layer to map m with name='Rainfall Heatmap' and show=False."

# YOUR CODE HERE

## Cell [8]: Add LayerControl

**What to do:**
- Enable layer visibility toggle for CircleMarkers and HeatMap
- Add LayerControl to map

In [ ]:
# Cell [8]: YOUR CODE HERE
# 1. Import LayerControl from folium
# 2. Add LayerControl(collapsed=False) to map m
# 3. This lets users toggle layers on/off

# AI Prompt suggestion:
# "Import LayerControl from folium.
#  Add LayerControl(collapsed=False) to map m.
#  Display the map."

# YOUR CODE HERE

## Cell [9]: Add Shelter Risk Popups

**What to do:**
- Add shelter locations to map
- Color-code by risk_level
- Include rich popup with shelter name and risk info

In [ ]:
# Cell [9]: YOUR CODE HERE
# 1. Loop through gdf_shelters and add Marker for each shelter
# 2. Color by risk_level: 'low'→blue, 'medium'→orange, 'high'→red
# 3. Create rich popup with HTML:
#    - Shelter name
#    - Risk level
#    - Terrain risk
#    - Mean elevation
#    - Max slope

# AI Prompt suggestion:
# "Loop through gdf_shelters. For each shelter, create an HTML popup:
#  <b>{name}</b><br>Risk: {risk_level}<br>Elevation: {mean_elevation}m<br>Max Slope: {max_slope}°
#  Add Marker with icon color based on risk_level.
#  Use folium.Icon(color='blue'/'orange'/'red', icon='info-sign')."

# YOUR CODE HERE

---

# Lab 1: CWA API → Folium Map (25 minutes)

**Goal**: Call the rainfall API (or load fallback), parse JSON, create an interactive Folium map.

> **Fallback**: If CWA API doesn't work, load `data/scenarios/fungwong_202511.json` instead. The structure is similar (both use `records.Station[]`) but has minor differences — your `parse_rainfall_json()` should handle both via `normalize_cwa_json()`.

**Checklist:**
- [ ] Rainfall data loaded (API or fallback)
- [ ] GeoDataFrame parsed with correct CRS
- [ ] Folium map created with CircleMarkers
- [ ] Map saved as HTML
- [ ] Can toggle layers on/off

### Lab 1 Step 1: Load Data (API or Fallback)

In [ ]:
# Lab 1 Step 1: YOUR CODE HERE
# 1. Read API_KEY from .env
# 2. Try: fetch_cwa_api(api_key)
# 3. If error or None: load 'data/scenarios/fungwong_202511.json'
# 4. Parse JSON → gdf_rainfall
# 5. Print shape, columns, CRS

# AI Prompt suggestion:
# "Try to fetch CWA API data. If it fails, load fungwong_202511.json from
#  data/scenarios/ directory. Parse either result with parse_rainfall_json.
#  Print the GeoDataFrame info."

# YOUR CODE HERE

In [13]:
def parse_rainfall_json(data):
    stations = normalize_cwa_json(data)
    records = []

    for s in stations:
        try:
            name = s.get("StationName", "Unknown")

            # 抓經緯度（更安全寫法）
            coords = s.get("GeoInfo", {}).get("Coordinates", [])
            lat, lon = None, None

            for c in coords:
                if isinstance(c, dict):
                    if "StationLatitude" in c:
                        lat = float(c["StationLatitude"])
                        lon = float(c["StationLongitude"])

            # 如果抓不到座標 → 跳過
            if lat is None or lon is None:
                continue

            # 抓 rainfall
            rainfall = None
            for r in s.get("RainfallElement", []):
                if r.get("ElementName") == "RAIN":
                    rainfall = float(r.get("ElementValue", -998))

            # 過濾無效值
            if rainfall is None or rainfall == -998:
                continue

            records.append({
                "station_name": name,
                "rainfall": rainfall,
                "geometry": Point(lon, lat)
            })

        except Exception as e:
            continue

    # 🔴 關鍵防呆：如果沒有資料
    if len(records) == 0:
        print("⚠️ 沒有成功解析任何雨量站，建立空 GeoDataFrame")

    # 正常建立
    gdf = gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs="EPSG:4326"
    )

    return gdf

In [ ]:
# Lab 1 Step 1
# 目標：
# 1. 從 .env 讀取 API_KEY
# 2. 先嘗試抓取 CWA API 雨量資料
# 3. 如果 API 失敗或回傳 None，就改讀 fallback JSON 檔
# 4. 用 parse_rainfall_json() 解析成 GeoDataFrame
# 5. 印出資料的 shape、columns、CRS

# 載入環境變數（如果前面已經 load_dotenv()，這行保留也沒問題）
load_dotenv()

# 從 .env 讀取 API key
# 這裡同時嘗試兩種常見命名，避免你的 .env 欄位名稱不同
api_key = os.getenv("CWA_API_KEY") or os.getenv("API_KEY")

# 先準備一個變數存放原始 JSON 資料
raw_data = None

# 設定 fallback 檔案路徑
fallback_path = "C:\Users\vicky\Desktop\NTU\GA 遙測與空間資訊之分析與應用\windsurf-project\week5\data\fungwong_202511.json"

# 先嘗試從 CWA API 抓資料
try:
    if api_key:
        print("正在嘗試抓取 CWA API 資料...")
        raw_data = fetch_cwa_api(api_key)
    else:
        print("找不到 API_KEY，改用 fallback 檔案。")
except Exception as e:
    print(f"CWA API 抓取失敗，原因：{e}")
    raw_data = None

# 如果 API 失敗、沒 key，或回傳 None，就改讀本機 JSON 檔
if raw_data is None:
    print(f"改讀 fallback 檔案：{fallback_path}")
    
with open(fallback_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)


# 將原始 JSON 解析成 GeoDataFrame
# 這裡假設你前面已經寫好 parse_rainfall_json()
gdf_rainfall = parse_rainfall_json(raw_data)

# 印出 GeoDataFrame 基本資訊
print("gdf_rainfall shape:", gdf_rainfall.shape)
print("gdf_rainfall columns:", list(gdf_rainfall.columns))
print("gdf_rainfall CRS:", gdf_rainfall.crs)

# 顯示前幾筆資料，方便快速檢查
gdf_rainfall.head()

正在嘗試抓取 CWA API 資料...
🌧️ Fetching rainfall data from CWA API...
📍 Endpoint: https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
🔑 API Key: CWA-6AA937...C2EF
✅ API request successful
📊 Retrieved unknown records
⏰ Data time: O-A0002-001


OSError: [Errno 22] Invalid argument: '**\\week5\\data\x0cungwong_202511.json'

### Lab 1 Step 2: Parse JSON → GeoDataFrame

In [ ]:
# Lab 1 Step 2: YOUR CODE HERE
# 1. Check gdf_rainfall has columns: rain_1hr, rain_3hr, rain_24hr
# 2. Check CRS is EPSG:4326
# 3. Display first 5 rows
# 4. Print statistics: min/max/mean rainfall

# AI Prompt suggestion:
# "Display gdf_rainfall.head(5) and gdf_rainfall.describe().
#  Check the CRS with gdf_rainfall.crs.
#  What are the highest rainfall stations?"

# YOUR CODE HERE

### Lab 1 Step 3: Build Folium Map + CircleMarkers

In [ ]:
# Lab 1 Step 3: YOUR CODE HERE
# 1. Create Folium map (reuse Cell [5] & [6])
# 2. Add CircleMarkers for rainfall stations
# 3. Add HeatMap layer
# 4. Add LayerControl
# 5. Display map

# AI Prompt suggestion:
# "Create a Folium map of Hualien with rainfall CircleMarkers and HeatMap.
#  Add LayerControl so users can toggle layers.
#  Display the map in the notebook."

# YOUR CODE HERE

### Lab 1 Step 4: Save Map as HTML

In [ ]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Save map to 'output/rainfall_map_week5.html'
# 2. Verify file was created
# 3. Print file size

# AI Prompt suggestion:
# "Save the map to 'output/rainfall_map_week5.html' using m.save().
#  Verify the file exists and print its size in KB."

# YOUR CODE HERE

---

# 🔬 Lab 2: Typhoon Fung-wong Simulation (15 minutes)

**Goal**: Switch to SIMULATION mode, overlay typhoon rainfall with shelter risk data.

> **Context**: It's 2025-11-11 14:00. Typhoon Fung-wong is hitting eastern Taiwan.
> Suao: 130.5mm/hr. Mataian Creek is forming a landslide dam.

**Checklist:**
- [ ] Simulation data loaded
- [ ] High-rainfall stations identified
- [ ] Shelters within 5km radius found
- [ ] Risk map created and saved

### Lab 2 Step 1: Load Simulation JSON + Filter High-Rain Stations

In [ ]:
# Lab 2 Step 1: YOUR CODE HERE
# 1. Load 'data/scenarios/fungwong_202511.json'
# 2. Parse with parse_rainfall_json()
# 3. Filter: rain_1hr > 30 mm/hr (heavy rain)
# 4. Print how many high-rain stations found
# 5. Find station with max rain_1hr (Suao should be 130.5mm/hr)

# AI Prompt suggestion:
# "Load and parse the Typhoon Fung-wong simulation JSON.
#  Filter for rain_1hr > 30 mm/hr. Count how many stations exceed this.
#  Find the station with highest rainfall. Is it Suao with ~130.5mm/hr?"

# YOUR CODE HERE

### Lab 2 Step 2: Spatial Join Rainfall with Shelters

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# 1. CRITICAL: Reproject gdf_rainfall to EPSG:3826 (same as shelters)
# 2. Filter high-rain stations (rain_1hr > 40mm)
# 3. Create 5km buffer around high-rain stations
# 4. Use gpd.sjoin(gdf_shelters, buffered_rain, how='left', predicate='within')
#    to find shelters inside the 5km impact zones
# 5. Flag shelters at risk and assign dynamic risk level

# AI Prompt suggestion:
# "Reproject gdf_rainfall to EPSG:3826 (meters).
#  Filter stations where rain_1hr > 40.
#  Create a 5000m buffer around each high-rain station geometry.
#  Make a new GeoDataFrame with the buffer as geometry.
#  Use gpd.sjoin with predicate='within' to find shelters inside buffers.
#  Apply risk logic: CRITICAL if rain > 80mm, URGENT if rain > 40mm
#  and terrain_risk == 'HIGH', WARNING otherwise."

# YOUR CODE HERE

### Lab 2 Step 3: Final Map + Save HTML

In [ ]:
# Lab 2 Step 3: YOUR CODE HERE
# 1. Create new Folium map (same center/zoom as Lab 1)
# 2. Add rainfall CircleMarkers (heavy rain = bigger, redder)
# 3. Add shelter Markers:
#    - Blue if low/medium risk
#    - Red if high_risk flag = True
# 4. Add HeatMap + LayerControl
# 5. Save to 'output/typhoon_fungwong_risk_map.html'
# 6. Display statistics: how many shelters are at risk?

# AI Prompt suggestion:
# "Create a new Folium map showing both rainfall and shelter risk.
#  Highlight shelters in red if they're within 5km of heavy rainfall.
#  Save to output/typhoon_fungwong_risk_map.html.
#  Print: 'X shelters are at HIGH RISK during Typhoon Fung-wong.'"

# YOUR CODE HERE

---

# 💭 My Reflection (fill in your answers)

### 1. How many lines of code did you change between LIVE and SIMULATION mode?

*Your answer:*

---

### 2. What happens if you forget to convert CRS before `sjoin`?

*Your answer:*

---

### 3. Why does CWA use -998 instead of NaN or null?

*Your answer:*

---

### 4. During Typhoon Fung-wong, which shelter would you evacuate first and why?

*Your answer:*

---

### 5. What challenges did you face in this lab? How did you solve them?

*Your answer:*